# SAFE-Alert — Baselines (16 cells riêng)

Mỗi cell = 1 baseline. Chạy xong cell nào → download zip của cell đó.

**Setup:** Add dataset `run-baseline-v2` → GPU T4 x2 → Chạy cell Setup trước, rồi chạy từng baseline.

**Lưu ý:** Mỗi cell mất ~5 phút load data + thời gian train.

In [ ]:
import os, sys, glob, subprocess, json, zipfile
import torch

WORK_DIR   = '/kaggle/working'
AI_SERVICE = '/kaggle/input/datasets/minhquan0706/run-baseline-v2/ai-service'
DATA_V2    = AI_SERVICE + '/training_data/v2'

assert os.path.exists(AI_SERVICE), f'AI_SERVICE not found: {AI_SERVICE}'
assert os.path.exists(DATA_V2),    f'DATA_V2 not found: {DATA_V2}'

PIPELINES = os.path.join(AI_SERVICE, 'app', 'v2', 'pipelines')
for p in [PIPELINES, os.path.join(AI_SERVICE,'app','v2'), AI_SERVICE]:
    sys.path.insert(0, p)
os.chdir(AI_SERVICE)

ARTIFACT_DIR = os.path.join(WORK_DIR, 'baselines')
os.makedirs(ARTIFACT_DIR, exist_ok=True)

SCRIPT = os.path.join(PIPELINES, 'run_baselines.py')
BASE_CMD = (
    f'python -u {SCRIPT}'
    f' --symbol BTCUSDT --horizon 1h --epochs 40 --batch_size 32'
    f' --data_path {DATA_V2} --embeddings_path {DATA_V2}'
    f' --artifact_dir {ARTIFACT_DIR}'
)

def run_baseline(name):
    out = os.path.join(ARTIFACT_DIR, f'result_{name}.json')
    cmd = f'{BASE_CMD} --baselines {name} --output {out}'
    print(f'>>> [{name}]', flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if os.path.exists(out):
        with open(out) as f:
            r = json.load(f)
        m = r.get(name, {})
        print(f'DONE {name}: F1={m.get("macro_f1",0):.3f}  Sharpe={m.get("alert_sharpe",0):.3f}', flush=True)
    else:
        print(f'ERROR: {out} not created', flush=True)
    return out

!pip install -q ta==0.11.0 vaderSentiment==3.3.2
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('Setup done. Chạy từng cell baseline bên dưới.')

In [ ]:
# BASELINE 1: market_only (~15 phút)
run_baseline('market_only')

In [ ]:
# BASELINE 2: all_news_fusion (~25 phút)
run_baseline('all_news_fusion')

In [ ]:
# BASELINE 3: always_alert (~2 phút, no training)
run_baseline('always_alert')

In [ ]:
# BASELINE 4: raw_prob_threshold (~15 phút)
run_baseline('raw_prob_threshold')

In [ ]:
# BASELINE 5: sentiment_market (~20 phút)
run_baseline('sentiment_market')

In [ ]:
# BASELINE 6: nsm_style (~30 phút)
run_baseline('nsm_style')

In [ ]:
# BASELINE 7: llm_factor (~20 phút)
run_baseline('llm_factor')

In [ ]:
# BASELINE 8: sep_style (~25 phút)
run_baseline('sep_style')

In [ ]:
# BASELINE 9: finin_style (~30 phút)
run_baseline('finin_style')

In [ ]:
# BASELINE 10: interleaved (~30 phút)
run_baseline('interleaved')

In [ ]:
# BASELINE 11: temperature_scaled (~15 phút)
run_baseline('temperature_scaled')

In [ ]:
# BASELINE 12: selective_forecasting (~30 phút)
run_baseline('selective_forecasting')

In [ ]:
# BASELINE 13: current_price_predictor (~5 phút)
run_baseline('current_price_predictor')

In [ ]:
# BASELINE 14: all_news_llm_expl (~30 phút)
run_baseline('all_news_llm_expl')

In [ ]:
# BASELINE 15: random_k_evidence (~25 phút)
run_baseline('random_k_evidence')

In [ ]:
# BASELINE 16: most_recent_k_evidence (~25 phút)
run_baseline('most_recent_k_evidence')

In [ ]:
# ===== MERGE tất cả → baseline_results.json =====
ALL_BASELINES = [
    'market_only','all_news_fusion','always_alert','raw_prob_threshold',
    'sentiment_market','nsm_style','llm_factor','sep_style',
    'finin_style','interleaved','temperature_scaled','selective_forecasting',
    'current_price_predictor','all_news_llm_expl','random_k_evidence','most_recent_k_evidence'
]

merged = {}
missing = []
for name in ALL_BASELINES:
    path = os.path.join(ARTIFACT_DIR, f'result_{name}.json')
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        if name in data:
            merged[name] = data[name]
            f1 = data[name].get('macro_f1', 0)
            sh = data[name].get('alert_sharpe', 0)
            print(f'[OK] {name:35s} F1={f1:.3f}  Sharpe={sh:.3f}')
    else:
        missing.append(name)
        print(f'[--] {name:35s} chưa chạy')

merged['_meta'] = {'protocol':'single_split_70_15_15','epochs':40,'n_baselines_run':len(merged)-0}
out = os.path.join(ARTIFACT_DIR, 'baseline_results.json')
with open(out, 'w') as f:
    json.dump(merged, f, indent=2)
print(f'\nMerged {len(merged)-1}/16 baselines → {out}')
if missing:
    print(f'Chưa có: {missing}')

# Bảng kết quả
print(f'\n  {"Baseline":35s}  {"F1":>6}  {"Sharpe":>7}  {"MCC":>6}')
print('  ' + '-'*60)
for name, m in merged.items():
    if name.startswith('_'): continue
    print(f'  {name:35s}  {m.get("macro_f1",0):.3f}  {m.get("alert_sharpe",0):7.3f}  {m.get("mcc",0):6.3f}')

zip_all = os.path.join(WORK_DIR, 'baselines_all.zip')
with zipfile.ZipFile(zip_all, 'w') as zf:
    zf.write(out, 'baseline_results.json')
    for name in ALL_BASELINES:
        p = os.path.join(ARTIFACT_DIR, f'result_{name}.json')
        if os.path.exists(p): zf.write(p, f'result_{name}.json')
print(f'\nDownload: {zip_all} ({os.path.getsize(zip_all)/1e3:.1f} KB)')